# EECE 6544: Introduction to Machine Learning and Pattern Recognition

## Assignment 02 · Telco Monthly Charge Prediction

Use this notebook to build a multivariable linear regression model that predicts a customer's monthly charge from their service bundle. Keep the work organized into the sections below, and replace each placeholder comment with your actual analysis, code, and short written interpretations.

## 1. Load and Inspect the Data

- Load `telco.csv`.
- Confirm the dataset shape is 7,043 rows.
- Verify that `target` is the continuous monthly charge target.
- Check that the service columns are clean 0/1 indicators with no missing values.
- Display `head()`, `shape`, `info()`, `describe()`, and missing-value counts.

In [61]:
import pandas as pd
df = pd.read_csv('telco.csv')
print(df.shape)
print(df.info())
print(df.describe())
print(df.isnull().sum())

(7043, 24)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 24 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   gender                                 7043 non-null   int64  
 1   SeniorCitizen                          7043 non-null   int64  
 2   Partner                                7043 non-null   int64  
 3   Dependents                             7043 non-null   int64  
 4   tenure                                 7043 non-null   int64  
 5   PhoneService                           7043 non-null   int64  
 6   MultipleLines                          7043 non-null   int64  
 7   OnlineSecurity                         7043 non-null   int64  
 8   OnlineBackup                           7043 non-null   int64  
 9   DeviceProtection                       7043 non-null   int64  
 10  TechSupport                            7043 non-null   int64 

## 2. Define the Variables

- Set the target as `MonthlyCharges` if you keep the original name, or confirm the renamed target column if you choose to rename it.
- Use the ten service indicators as predictors:
  - `PhoneService`
  - `MultipleLines`
  - `OnlineSecurity`
  - `OnlineBackup`
  - `DeviceProtection`
  - `TechSupport`
  - `StreamingTV`
  - `StreamingMovies`
  - `InternetService_Fiber optic`
  - `InternetService_No`
- Confirm the predictors are numeric and ready for regression.

In [62]:
# Define the target and the exact service predictors required by the assignment
feature_columns = [
    'PhoneService',
    'MultipleLines',
    'OnlineSecurity',
    'OnlineBackup',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies',
    'InternetService_Fiber optic',
    'InternetService_No',
]

X = df[feature_columns].copy()
MonthlyCharges = df['target'].copy()

print('\nPredictor shape:', X.shape)
print('MonthlyCharges shape:', MonthlyCharges.shape)
print('\nPredictor dtypes:')
print(X.dtypes)


Predictor shape: (7043, 10)
MonthlyCharges shape: (7043,)

Predictor dtypes:
PhoneService                   int64
MultipleLines                  int64
OnlineSecurity                 int64
OnlineBackup                   int64
DeviceProtection               int64
TechSupport                    int64
StreamingTV                    int64
StreamingMovies                int64
InternetService_Fiber optic    int64
InternetService_No             int64
dtype: object


## 3. Split the Data

In [63]:
from sklearn.model_selection import train_test_split
# split the data into training and test sets
X_train, X_test, Y_train, Y_test = train_test_split(X, MonthlyCharges, test_size=0.2, random_state=42)

## 4. Fit the Multivariable Linear Regression Model

In [64]:
# import and fit LinearRegression
from sklearn.linear_model import LinearRegression
model = LinearRegression()
# train the model on X_train and y_train
model.fit(X_train, Y_train)

LinearRegression()

## 5. Report the Model Parameters

In [65]:
print('Intercept:', model.intercept_)
pd.Series(model.coef_, index=X.columns)

Intercept: 24.969058841322493


PhoneService                   20.036039
MultipleLines                   5.012293
OnlineSecurity                  5.045880
OnlineBackup                    4.979713
DeviceProtection                5.011923
TechSupport                     5.028676
StreamingTV                     9.980786
StreamingMovies                 9.946628
InternetService_Fiber optic    24.951892
InternetService_No            -25.047916
dtype: float64

## 6. Evaluate on the Test Set

In [66]:
# generate predictions on the test set
from sklearn.metrics import mean_absolute_error, root_mean_squared_error


Y_pred = model.predict(X_test)
# calculate R2, MAE, and RMSE
r2 = model.score(X_test, Y_test)
mae = mean_absolute_error(Y_test, Y_pred)
rmse = root_mean_squared_error(Y_test, Y_pred)
print(f'R2: {r2:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}')

R2: 0.9988, MAE: 0.7886, RMSE: 1.0504


## 7. Compare to the Single-Variable Baseline

In [68]:
df['AddonsCount'] = df[X.columns].sum(axis=1)
X_single = df[['AddonsCount']].copy()
Y_single = df['target'].copy()
X_train_single, X_test_single, Y_train_single, Y_test_single = train_test_split(X_single, Y_single, test_size=0.2, random_state=42)
model_single = LinearRegression()
model_single.fit(X_train_single, Y_train_single)
Y_pred_single = model_single.predict(X_test_single)
print('Single-variable model performance:')
r2 = model_single.score(X_test_single, Y_test_single)
mae = mean_absolute_error(Y_test_single, Y_pred_single)
rmse = root_mean_squared_error(Y_test_single, Y_pred_single)
print(f'R2: {r2:.4f}, MAE: {mae:.4f}, RMSE: {rmse:.4f}')

Single-variable model performance:
R2: 0.7009, MAE: 14.2369, RMSE: 16.4551


## 8. Interpret the Results

The model has an intercept of 24.97 which means that without add-ons the base-plan charge of about £24.97.

The coefficient can be read as the approximate monthly price contribution of that service. Some services have a positive effect on the monthly charge, while others have a negative effect. For example, having fiber optic internet adds about £24.95 to the monthly charge, while not having internet reduces the bill by about £25.05.Overall, the learned prices match the expected Telco price structure, with add-ons increasing the monthly charge and the absence of services reducing it.

## 9. Make a Prediction for a New Customer

In [69]:
X_pred = pd.DataFrame({
    'PhoneService': [1],
    'MultipleLines': [0],
    'OnlineSecurity': [1],
    'OnlineBackup': [0],
    'DeviceProtection': [1],
    'TechSupport': [0],
    'StreamingTV': [1],
    'StreamingMovies': [0],
    'InternetService_Fiber optic': [1],
    'InternetService_No': [0],
})
Y_pred_new = model.predict(X_pred)
print(f'Predicted monthly charge for the new customer: £{Y_pred_new[0]:.2f}')

Predicted monthly charge for the new customer: £90.00
